# Competencia 01 — v3: pipeline secuencial y adaptativo

Este notebook **no contiene el pipeline**: lo ejecuta y muestra lo que decidió. La lógica vive en
`pipeline/p10_adaptativo.py`, por las mismas tres razones de `cuaderno_pipeline.ipynb`: las corridas largas
van con `nohup` (regla 11), los gates tienen que poder matar el proceso y la entrega tiene que ser
reproducible desde una orden.

| Archivo | Qué hace |
|---|---|
| `pipeline/p10_adaptativo.py` | El pipeline: contexto, banco de medición, etapas 1 a 4 |
| `pipeline/p11_sintetico.py` | Datos sintéticos con la forma del problema real (y verdad oculta de agosto) |
| `pipeline/p00_config.py` | Fuente única de ganancia, meses, folds y semillas (no se duplica nada) |

## Qué significa "adaptativo" acá

Cada etapa **mide** y lo medido **decide** qué hace la siguiente. Toda decisión queda en
`ContextoPipeline` con su evidencia numérica y se escribe a disco en el momento (`decisiones.jsonl`,
`estado.json`). Si el proceso muere, lo decidido no se pierde, y al relanzarlo el banco reusa las
mediciones cacheadas.

| Etapa | Punto de decisión | Qué la dispara |
|---|---|---|
| 1. Ingesta y diagnóstico | Gates (claves, granularidad, gap F1, scoring sin etiqueta F2, columnas prohibidas F4) | Cualquier falla **aborta** (código de salida 2) |
| | Constantes → se eliminan | `nunique == 1` |
| | Variables rotas → **candidato** para la etapa 2 | ≥99 % ceros/nulos en un mes que antes era normal |
| | Deriva (PSI > 0,25) → **ablación** para la etapa 2 | PSI contra el primer mes |
| | Desbalance → módulo de undersampling | Prevalencia < 5 % (la real es 0,62 %) |
| 2. Ingeniería iterativa | Gate F5 exacto (¿lee el futuro?) | Construir truncando en 202105 y comparar |
| | ¿Hay historia en el train del fold? | Si no, el fold no puede medir el bloque |
| | Acepta / rechaza / poda | Wilcoxon apareado por semilla |
| | Reintento corregido | Solo ante fallas **técnicas** |
| 3. Optimización | Espacio de búsqueda según dimensionalidad | `p` features y `n` filas resultantes |
| | Poda de trials | Mediana de Optuna + piso duro en la iteración 50 |
| | Confirmación | En el fold que la búsqueda **no** vio, con Bonferroni |
| 4. Corte y entrega | ¿Reemplazar 0,025? | Una regla elegida en un mes y medida en el otro, en los dos sentidos |

## Qué heredé de `competencia.ipynb` y del Experimento 01 (y por qué no es la plantilla tal cual)

La plantilla pedía tres cosas que en **este** problema son errores ya cometidos o ya medidos:

1. **"Validación cruzada Out-of-Fold"**: k-fold aleatorio mezcla meses y clientes. Lo reemplazan los dos folds
   temporales con gap 2 (`[202103]→202105`, `[202103,202104]→202106`). El primer baseline de
   `competencia.ipynb` tenía gap 1 y daba 311 M inflados.
2. **"Reinyectar la métrica para corregir el próximo intento"**: si un bloque no mejora y lo retoco mirando
   la validación hasta que mejore, estoy eligiendo y midiendo en el mismo lugar (fuga F6, que el Exp01 midió
   en +35,9 M en junio para el corte). El pipeline **sí** reintenta, pero solo ante fallas técnicas: una
   columna que no existe, o una fuga de futuro que tiene corrección (rank global → rank por mes).
3. **"Ajustar el umbral según costo/beneficio"**: el umbral de costo/beneficio ya es 0,025 = 27.500 / 1.100.000.
   El Exp01 mostró que el óptimo no viaja entre meses (k óptimo +71 % de mayo a junio). La etapa 4 vuelve a
   correr esa prueba automáticamente y solo cambia el corte si otra regla gana en los dos sentidos.

Otros aprendizajes que el pipeline aplica por construcción:

- **20 semillas apareadas**. El desvío entre semillas en junio es 13,18 M. Con 5 semillas solo se resuelven
  efectos de ~20 M, y las decisiones de bloques de `competencia.ipynb` se tomaron con 5 semillas, un mes y sin test.
- **150 iteraciones fijas**, sin early stopping: mayo pide 300 y junio 150, ambos con p < 0,01.
- **Ganancia desde `clase_ternaria`**, nunca desde la binaria.
- **Undersampling como palanca de cómputo**, con prueba de **no inferioridad** y corrección de la probabilidad
  antes de cortar en 0,025.

### Un hallazgo nuevo que salió al armar esto

Con 6 meses, **ningún lag tiene historia en el train del fold de mayo** (202103 no tiene febrero), y `lag2` o
más no la tiene en ningún fold. O sea, el bloque lag1 de las 238 features solo pudo validarse en junio, y la
regla "gana en los dos folds" es imposible de cumplir para cualquier feature temporal. El pipeline lo detecta,
mide el bloque solo en los folds donde existe historia y marca la decisión como **EVIDENCIA PARCIAL**.
En la Última Competencia, con 18 meses, esto desaparece.

In [ ]:
import sys, json, glob, warnings
from pathlib import Path
import pandas as pd

warnings.filterwarnings("ignore")
PROYECTO = Path.cwd()
sys.path.insert(0, str(PROYECTO / "pipeline"))

import importlib
import p10_adaptativo as p10
importlib.reload(p10)  # por si se editó el .py con el kernel abierto

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

print("ganancia por acierto:", f"${p10.cfg.GANANCIA_BAJA2:,}", "| costo:", f"${p10.cfg.COSTO_ESTIMULO:,}",
      "| umbral:", p10.cfg.UMBRAL_TEORICO)
print("folds:", [(f["nombre"], f["train"], f["validacion"]) for f in p10.cfg.FOLDS])

## Las decisiones de modelado, en un solo lugar

Todo lo que está en este diccionario es **decisión de modelado**, no infraestructura: alpha, mejora mínima,
semillas, umbral de desbalance, margen de no inferioridad, rangos de búsqueda. Lo tengo que poder defender
el 10 de diciembre.

In [ ]:
pd.Series({k: v for k, v in p10.DECISIONES_DE_MODELADO.items()
           if k not in ("semillas_medicion",)}, name="valor").to_frame()

## Demo con datos sintéticos (~1 minuto)

Los datos sintéticos (`p11_sintetico.py`) tienen la misma forma que los reales: clase ternaria deducida por
presencia, prevalencia de ~0,7 %, inflación en los montos, una variable rota en 202105, una constante, clientes
con huecos y una **tendencia de deterioro** por cliente, que es la señal que la historia puede captar.
Ventaja sobre los datos reales: agosto tiene **verdad oculta**, así que al final se puede auditar la entrega.

El catálogo sintético incluye además tres candidatos que existen solo para ejercitar los caminos de error:

- `trampa_lead1` lee el mes siguiente. El gate F5 la descarta sin corrección posible.
- `rank_global` rankea contra toda la tabla. Es fuga, pero se corrige a rank por mes.
- `lag3_con_typo` pide una columna que no existe. Se reintenta sin ella.

In [ ]:
ctx = p10.correr("sintetico")

### El log de decisiones

In [ ]:
ctx.tabla_decisiones()[["etapa", "punto", "resultado", "motivo"]]

### Bloques de la etapa 2, con la evidencia por fold

`delta` es la diferencia media apareada por semilla contra el incumbente de ese momento. `p` es el p-valor
del Wilcoxon unilateral. `a_favor` cuenta en cuántas semillas ganó el candidato.

In [ ]:
def tabla_bloques(ctx):
    filas = []
    for estado, lista in (("aprobado", ctx.bloques_aprobados), ("rechazado", ctx.bloques_rechazados)):
        for b in lista:
            v = b.get("veredicto")
            if not v:
                filas.append({"bloque": b["bloque"], "estado": f"falla: {b['motivo']}"})
                continue
            for f, r in v["por_fold"].items():
                filas.append({"bloque": b["bloque"], "estado": estado, "fold": f,
                              "podado": v["podado"], "semillas": r["n"],
                              "delta_M": r["delta"] / 1e6, "delta_rel": r["delta_rel"],
                              "p": r["p_mejor"], "a_favor": r["semillas_a_favor"]})
    return pd.DataFrame(filas)

tabla_bloques(ctx)

### Cómo evolucionó el incumbente, y auditoría de la entrega contra la verdad de agosto

In [ ]:
for etapa, r in ctx.metricas.items():
    print(f"{etapa:9s}", p10._texto_resumen(r))
print()
print("historial:", *ctx.historial_incumbentes, sep="\n  ")
print()
e = ctx.entrega
print(f"entrega: {e['estimulados']:,} estimulados ({e['fraccion']:.2%}) con {e['regla']}")
print(f"ganancia real en agosto (sintético): {e['ganancia_real_sintetica']/1e6:,.2f} M "
      f"de {e['ganancia_optima_sintetica']/1e6:,.2f} M posibles")
print("salidas en:", ctx.carpeta)

**Cómo leer la demo.** En el sintético, el bloque `lag1_delta1` mejora cerca de +5 % en junio, pero con 10
semillas no llega a p < 0,05 y **queda afuera**. No es un bug: es la misma falta de poder que el Exp01 encontró
con 5 semillas. La etapa 3 sí encuentra hiperparámetros mejores que el baseline sintético, que es arbitrario,
y los **confirma en junio**, un fold que la búsqueda no vio.

## Corrida real

El modo real usa la base cacheada del Experimento 01 (`experimentos/base_modelado.pkl`, 238 features), con el
trial 25 congelado como incumbente inicial. **No se corre desde el notebook** (regla 11): dura ~1 hora.

```bash
# 1. smoke test: 2 semillas y 6 trials, ~3 minutos. Verifica que no se rompe en la línea 300.
python3 -u pipeline/p10_adaptativo.py --modo real --max-semillas 2 --optuna-trials 6

# 2. corrida completa, desacoplada de la sesión
nohup python3 -u pipeline/p10_adaptativo.py --modo real > pipeline/salidas/p10_real.log 2>&1 &
tail -f pipeline/salidas/p10_real.log
```

Si se corta, al relanzarla el banco reusa todo lo ya medido (`pipeline/salidas/p10_cache_real/`).
Cada corrida escribe en su propia carpeta con fecha y hora: nada se pisa.

La celda siguiente lee la **última** corrida real que haya en disco.

In [ ]:
corridas = sorted(glob.glob(str(PROYECTO / "pipeline/salidas/p10_real_*/estado.json")))
if not corridas:
    print("todavía no hay corridas reales")
else:
    estado = json.load(open(corridas[-1]))
    print("corrida:", Path(corridas[-1]).parent.name, "| etapas:", estado["etapas_completas"])
    print("semillas de medición:", len(estado["decisiones_de_modelado"]["semillas_medicion"]))
    print("incumbente final:", estado["incumbente"]["nombre"], f"({estado['incumbente']['n_features']} features)")
    print("entrega:", estado["entrega"].get("archivo"), "-", estado["entrega"].get("estimulados"), "estimulados")
    display(pd.DataFrame(estado["decisiones"])[["etapa", "punto", "resultado", "motivo"]])

## Para el examen

Infraestructura (me apoyo en Claude sin culpa): el contexto persistente, el cache del banco, el manejo de
excepciones, `nohup`, la escritura atómica del estado.

Decisiones de modelado (son mías y las tengo que defender):

1. ¿Por qué una feature temporal no puede cumplir "gana en los dos folds" con 6 meses, y qué hace el pipeline con eso?
2. ¿Por qué el undersampling se evalúa por **no inferioridad** y no por superioridad? ¿Qué pasa si corto en 0,025
   sin corregir la probabilidad?
3. ¿Por qué el pipeline reintenta un bloque con una columna faltante pero **no** uno que empeoró la ganancia?
4. ¿Por qué Optuna elige en mayo y confirma en junio, y por qué se usa Bonferroni en la confirmación?
5. ¿Qué prueba exactamente el gate F5, y por qué es una prueba y no un indicio?

> **Explicá en 60 segundos, sin mirar:** ¿por qué la etapa 4 casi nunca cambia el corte 0,025, aunque 0,025 no sea el óptimo?